<img src="https://github.com/IKNL/guidelines/blob/master/resources/logos/iknl_nl.png?raw=true" width=200 align="right">

# Data Preparation
**Phase 1: Fake OMOP data in UPM and IKNL**


In [ ]:
# The data preparation step in the RAVEN UI requires the *summary* method to be run
# in the vantage6 network.
#
# AUTHENTICATION
# --------------------------------------------------------------------------------------
# In this notebook we first authenticate to obtain a JWT token which can be used for the
# successive calls. This authentication is *not* coupled to the CERTH Keycloak. The
# token returned is set to expire after 5 days for this demo so we do not need to
# re-authenticate to often. I suggest to hard-code the token in the RAVEN UI/API for
# now.
#
# SUMMARY ALGORITHM
# --------------------------------------------------------------------------------------
# The summary algorithm computes a lot of descriptive statistics. I suggest to have a
# brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-summary-py/docs/v6-summary-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a two (federated-)step algorithm:
#
# 1. Call `summary_per_data_station`
# 2. Call `variance_per_data_station`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the summary statistics for
# the entire federated dataset. In IDEA4RC, the summary statistics per data station are
# also required. So in this notebook we go through the following steps to obtain both
# the *global* (from the central part) and the *local* (from the
# `summary_per_data_station` call) summary statistics:
#
# 1. Create a new vantage6 task to execute the *summary* method (central part). This
#    central part will start the tasks `summary_per_data_station` and
#    `variance_per_data_station` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* summary statistics from the central part (the main call)
# 4. Retrieve the *local* summary statistics from the data stations (the
#    `summary_per_data_station` call that was made by the central part)
#


In [2]:
import requests
import json
import base64

from vantage6.client import UserClient

## Authentication


In [3]:
client = UserClient(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es:443/server",
    auth_url="https://vantage6-auth.orchestrator.idea.lst.tfo.upm.es:443",
    auth_client="public_client",
    auth_realm="vantage6",
    log_level="INFO"
)
# You can authenticate using the user `itziar` and the password that I've send to you.
client.authenticate()

# Set the headers for the other requests
headers = {
    "Authorization": f"Bearer {client._access_token}"
}

# Print the server version
print("Server version: ", client.util.get_server_version())

 Welcome to
                  _                     __  
                 | |                   / /  
__   ____ _ _ __ | |_ __ _  __ _  ___ / /_  
\ \ / / _` | '_ \| __/ _` |/ _` |/ _ \ '_ \ 
 \ V / (_| | | | | || (_| | (_| |  __/ (_) |
  \_/ \__,_|_| |_|\__\__,_|\__, |\___|\___/ 
                            __/ |           
                           |___/            

 --> Join us on Discord! https://discord.gg/rwRvwyK
 --> Docs: https://docs.vantage6.ai
 --> Blog: https://vantage6.ai
------------------------------------------------------------
Cite us!
If you publish your findings obtained using vantage6, 
please cite the proper sources as mentioned in:
https://vantage6.ai/vantage6/references
------------------------------------------------------------
Opening browser for login


127.0.0.1 - - [18/Nov/2025 10:38:33] "GET /callback?state=state&session_state=fcab9cc2-e144-4b80-9d57-c954a3d9158f&iss=https%3A%2F%2Fvantage6-auth.orchestrator.idea.lst.tfo.upm.es%2Frealms%2Fvantage6&code=23b7823c-4128-4440-8009-a9da3eef7156.fcab9cc2-e144-4b80-9d57-c954a3d9158f.fcb015fe-d5b0-4a7b-b609-7d87a3b72f3e HTTP/1.1" 200 -


 --> Succesfully authenticated
 --> Name: admin (id=1)
 --> Organization: root (id=1)
Server version:  {'version': '5.0.0a43'}


In [ ]:
headers

## Summary Algorithm

In [5]:
# The vantage6 server requires a certain payload to the request. It requires:
#
# * The STUDY_ID (which is implicit also defining the COLLABORATION_ID)
# * The SESSION_ID
# * The ORG_IDS (all the organizations that should be included in the analysis)
# * IMAGE (the docker image to use that contains the summary algorithm)
# * METHOD (the method to execute)
#
# For this demo (in phase 1) we hard-code all of these values.
#
#
# This study is part of collaboration 2, and consists of UPM and IKNL
STUDY_ID = 3
#
# A session that is already part of the study is 2
SESSION_ID = 2
#
# The image to use is the latest version of the sessions algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm
METHOD = "summary"
#
# Organization IDs for UPM and IKNL
UPM_ORG_ID = 3
IKNL_ORG_ID = 1
ORG_IDS = [UPM_ORG_ID, IKNL_ORG_ID]

In [6]:
# Besides that the vantage6 server expect a certain payload the algorithm also
# expects certain input:
#
# * DATAFRAME_IDS (the IDs of the dataframes to include in the analysis)
# * VARIABLES (the variables to include in the analysis)
# * NUMERIC_VARIABLES (the numeric variables that are included in the analysis, should
#   be a subset of VARIABLES)
#
#
# These dataframes are already created and part of the session/study.
pelvis = 76 # Pelvis cohort ID
rps_pelvis = 77 # RPS+Pelvis cohort ID
rps = 78 # RPS cohort ID
DATAFRAME_IDS = [pelvis, rps_pelvis, rps]
#
# These are the variables that currently exist in the dataframes.
VARIABLES = [
    "age", # num
    "tumor_size", # num
    "histology", # cat
    "sex", # cat
    "fnclcc_grade", # cat
    "multifocality", # cat
    "completeness_of_resection", # cat
    "tumor_rupture", # cat
    "pre_operative_chemo", # cat
    "post_operative_chemo", # cat
    "pre_operative_radio", # cat
    "post_operative_radio", # cat
    "local_recurrence", # cat
    "distant_metastasis", # cat
    "status", # cat
]
#
# From these variables, the numeric variables are:
NUMERIC_VARIABLES = [
    "age",
    "tumor_size"
]
#
# The input for the central part of the algorithm is the following.
#
# * The variables to include in the analysis
# * The numeric variables to include in the analysis
# * The organizations to include in the analysis
#
# These require to be encoded as follows:
org_input = [
    {
        "id": UPM_ORG_ID, # Central task is executed by UPM
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "columns": VARIABLES,
                    "numeric_columns": NUMERIC_VARIABLES,
                    "organizations_to_include": ORG_IDS
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

In [7]:
# Then the full payload of both the server requirements and the algorithm input is:
payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}

In [8]:
# -------------------------------------------------------------------------------------
#|  You can simply copy the following payload into the RAVEN UI for now. In the future |
#|  some of the values that we now hard-coded should be replaced by the RAVEN UI.      |
# -------------------------------------------------------------------------------------
payload

{'name': 'Human-readable name of the task',
 'image': 'harbor2.vantage6.ai/idea4rc/analytics:latest',
 'description': 'Description of the task',
 'action': 'central_compute',
 'method': 'summary',
 'organizations': [{'id': 3,
   'arguments': 'eyJjb2x1bW5zIjogWyJhZ2UiLCAidHVtb3Jfc2l6ZSIsICJoaXN0b2xvZ3kiLCAic2V4IiwgImZuY2xjY19ncmFkZSIsICJtdWx0aWZvY2FsaXR5IiwgImNvbXBsZXRlbmVzc19vZl9yZXNlY3Rpb24iLCAidHVtb3JfcnVwdHVyZSIsICJwcmVfb3BlcmF0aXZlX2NoZW1vIiwgInBvc3Rfb3BlcmF0aXZlX2NoZW1vIiwgInByZV9vcGVyYXRpdmVfcmFkaW8iLCAicG9zdF9vcGVyYXRpdmVfcmFkaW8iLCAibG9jYWxfcmVjdXJyZW5jZSIsICJkaXN0YW50X21ldGFzdGFzaXMiLCAic3RhdHVzIl0sICJudW1lcmljX2NvbHVtbnMiOiBbImFnZSIsICJ0dW1vcl9zaXplIl0sICJvcmdhbml6YXRpb25zX3RvX2luY2x1ZGUiOiBbMywgMV19'}],
 'databases': [[{'type': 'dataframe', 'dataframe_id': 76},
   {'type': 'dataframe', 'dataframe_id': 77},
   {'type': 'dataframe', 'dataframe_id': 78}]],
 'session_id': 2,
 'study_id': 3}

In [9]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

{'job_id': 83,
 'algorithm_store': None,
 'description': 'Description of the task',
 'runs': '/server/run?task_id=231',
 'finished_at': None,
 'study': {'id': 3,
  'link': '/server/study/3',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'databases': [{'label': None,
   'type': 'dataframe',
   'dataframe_id': 76,
   'dataframe_name': 'Pelvis',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 77,
   'dataframe_name': 'Pelvis_RPS',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 78,
   'dataframe_name': 'RPS',
   'position': 0}],
 'dataframe': None,
 'required_by': [],
 'status': 'awaiting',
 'created_at': '2025-11-18T09:23:16.060188',
 'id': 231,
 'depends_on': [],
 'init_org': {'id': 1,
  'link': '/server/organization/1',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'parent': None,
 'collaboration': {'id': 2,
  'link': '/server/collaboration/2',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'children': '/server/task?parent_id=231',


In [28]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

'completed'

In [29]:
# Get the results of the (central) task, thus the *global* summary statistics.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

{'RPS': {'numeric': {'age': {'count': 628.0,
    'min': 18.0,
    'max': 75.0,
    'missing': 0.0,
    'sum': 28552.0,
    'median': {'root': 45.0, 'UPM': 45.0},
    'q_25': {'root': 32.0, 'UPM': 32.0},
    'q_75': {'root': 60.0, 'UPM': 60.0},
    'mean': 45.46496815286624,
    'std': 16.719480720559844},
   'tumor_size': {'count': 572.0,
    'min': 1.1,
    'max': 10.0,
    'missing': 56.0,
    'sum': 3084.2,
    'median': {'root': 5.300000000000001, 'UPM': 5.300000000000001},
    'q_25': {'root': 3.025, 'UPM': 3.025},
    'q_75': {'root': 7.875, 'UPM': 7.875},
    'mean': 5.391958041958041,
    'std': 2.6609003279255177}},
  'categorical': {'multifocality': {'count': 628, 'missing': 0},
   'pre_operative_radio': {'count': 628.0, 'missing': 0.0},
   'fnclcc_grade': {'count': 628, 'missing': 0},
   'status': {'count': 628, 'missing': 0},
   'pre_operative_chemo': {'count': 628.0, 'missing': 0.0},
   'post_operative_radio': {'count': 628.0, 'missing': 0.0},
   'tumor_rupture': {'count':

In [30]:
# The central task created two subtasks, see the swimlane diagram (reference above). We
# first need to retrieve the subtask IDs and then we can obtain the results from this.
# We do not need to poll until the subtasks are finished, as the central part is
# finished.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task?parent_id={TASK_ID}",
    headers=headers,
)
# We expect two subtasks for the summary algorithm, we are only interested in the one
# that triggers the `summary_per_data_station` method. I obtained the subtask ID here
# by looking at the method name, it is also possible to just obtain all the subtask IDs
# and then picking the lowest number (as `summary_per_data_station` is the first method
# to be called).
for task in response.json()["data"]:
    if task["method"] == "summary_per_data_station":
        SUBTASK_ID = task["id"]
        break
SUBTASK_ID


232

In [31]:
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={SUBTASK_ID}",
    headers=headers,
)
# Since this demo includes 2 organizations, we expect 2 results. Each for each center.
# The `organization_id` is included in the result, so we can easily identify the source.
for result in response.json()["data"]:
    print(json.loads(base64.b64decode(result["result"]).decode("UTF-8")))


{'Pelvis': {'numeric': {'age': {'count': 286.0, 'min': 18.0, 'max': 75.0, 'missing': 0.0, 'sum': 13595.0, 'median': 46.5, 'q_25': 36.0, 'q_75': 62.0}, 'tumor_size': {'count': 251.0, 'min': 1.0, 'max': 9.9, 'missing': 35.0, 'sum': 1375.3, 'median': 5.5, 'q_25': 3.1, 'q_75': 7.8}}, 'categorical': {'multifocality': {'count': 286, 'missing': 0}, 'pre_operative_radio': {'count': 286.0, 'missing': 0.0}, 'fnclcc_grade': {'count': 286, 'missing': 0}, 'status': {'count': 286, 'missing': 0}, 'pre_operative_chemo': {'count': 286.0, 'missing': 0.0}, 'post_operative_radio': {'count': 286.0, 'missing': 0.0}, 'tumor_rupture': {'count': 286.0, 'missing': 0.0}, 'local_recurrence': {'count': 286.0, 'missing': 0.0}, 'completeness_of_resection': {'count': 286, 'missing': 0}, 'histology': {'count': 286, 'missing': 0}, 'distant_metastasis': {'count': 286.0, 'missing': 0.0}, 'post_operative_chemo': {'count': 286.0, 'missing': 0.0}, 'sex': {'count': 286, 'missing': 0}}, 'num_complete_rows_per_node': 251, 'cou